# Artists vs. Diffusion Models — Experiment 1: Loss-Based MIA

Tests whether SD 1.5 (trained on Rutkowski) and SD 2.1 (Rutkowski excluded) show a detectable loss gap between his images and Bowater's (control).

**Hardware:** T4 GPU. **Runtime:** ~45 min.

---
## Cell 1: Colab Setup

In [17]:
# Colab Setup
import os
import sys

# Prevent transformers from importing TensorFlow — avoids OwnedIterator crash on Colab
os.environ['TRANSFORMERS_NO_TF'] = '1'

ON_COLAB = 'google.colab' in str(get_ipython())

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PATH = "/content/drive/Othercomputers/laptop/research/diffusion_art"
else:
    possible_paths = [
        "/mnt/c/research/diffusion_art",
        os.path.expanduser("~/research/diffusion_art"),
        ".",
    ]
    PATH = None
    for p in possible_paths:
        if os.path.exists(os.path.join(p, "assets")):
            PATH = os.path.abspath(p)
            break
    if PATH is None:
        raise RuntimeError("Could not find diffusion_art folder. Set PATH manually.")

print(f"ON_COLAB: {ON_COLAB}")
print(f"PATH: {PATH}")

os.chdir(PATH)
sys.path.insert(0, PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ON_COLAB: True
PATH: /content/drive/Othercomputers/laptop/research/diffusion_art


---
## Cell 2: Install Dependencies

In [ ]:
# Install dependencies
# pyarrow 17+ is required — first version compatible with numpy 2.x.
!pip install -q "pyarrow>=17.0"
!pip install -q --upgrade diffusers transformers accelerate
!pip install -q torch torchvision pillow
!pip install -q plotly
!pip install -q open_clip_torch ftfy regex

# CSD repo for style similarity
if not os.path.exists("csd_repo"):
    !git clone -q https://github.com/learn2phoenix/CSD.git csd_repo
sys.path.insert(0, os.path.join(PATH, "csd_repo"))

print("Dependencies installed.")
print()
print("IMPORTANT: Go to Runtime → Restart session, then run all cells from the top.")


---
## Cell 3: Global Imports and Config

In [ ]:
# Pre-initialize TF so its proto descriptors are registered before
# diffusers/transformers import CLIP — prevents 'already registered' conflict.
import tensorflow as tf  # noqa: F401

import os, sys, json, glob
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

# ROC curve and AUC in pure numpy — avoids sklearn → pandas → pyarrow conflict
def roc_curve_np(y_true, y_score):
    desc = np.argsort(y_score)[::-1]
    y_true = np.asarray(y_true)[desc]
    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos
    tpr = np.concatenate([[0], np.cumsum(y_true) / n_pos])
    fpr = np.concatenate([[0], np.cumsum(1 - y_true) / n_neg])
    return fpr, tpr

def auc_np(fpr, tpr):
    return float(np.trapz(tpr, fpr))

# Directories
ASSETS_DIR    = os.path.join(PATH, "assets")
PORTFOLIO_DIR = os.path.join(ASSETS_DIR, "portfolio")
CONTROL_DIR   = os.path.join(ASSETS_DIR, "control")
GENERATED_DIR = os.path.join(ASSETS_DIR, "generated")
LAION_DIR     = os.path.join(ASSETS_DIR, "laion_matches")
GEMINI_DIR    = os.path.join(ASSETS_DIR, "gemini")
FIGURES_DIR   = os.path.join(ASSETS_DIR, "figures")
RESULTS_DIR   = os.path.join(PATH, "results")

for d in [GENERATED_DIR, LAION_DIR, GEMINI_DIR, FIGURES_DIR, RESULTS_DIR,
          os.path.join(GENERATED_DIR, "sd15_style"),
          os.path.join(GENERATED_DIR, "sd21_style"),
          os.path.join(GENERATED_DIR, "sd15_caption"),
          os.path.join(GENERATED_DIR, "mia")]:
    os.makedirs(d, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Results accumulator
results = {}

# Plotly template
pio.templates.default = "plotly_white"

def save_fig(fig, path):
    base = path.replace(".png", "")
    pio.write_json(fig, base + ".json")
    fig.write_html(base + ".html", include_plotlyjs="cdn")

print("Imports OK.")


---
## Cell 4: Load Artist Portfolios

**Before running:** Add images to:
- `assets/portfolio/` — 10-15 Rutkowski paintings (PNG or JPG)
- `assets/control/` — 10-15 Charlie Bowater paintings (PNG or JPG)

Good sources: ArtStation (artstation.com/rutkowski, artstation.com/charliebowater)

In [20]:
def load_images(directory, size=(512, 512)):
    """Load all images from a directory, return as list of PIL Images."""
    extensions = ["*.png", "*.jpg", "*.jpeg", "*.webp",
                  "*.PNG", "*.JPG", "*.JPEG", "*.WEBP"]
    paths = []
    for ext in extensions:
        paths.extend(glob.glob(os.path.join(directory, ext)))
    paths = sorted(set(paths))  # deduplicate in case of case-insensitive FS

    images = []
    skipped = []
    for p in paths:
        try:
            img = Image.open(p).convert("RGB")
            if size:
                img = img.resize(size, Image.LANCZOS)
            images.append({"path": p, "name": os.path.basename(p), "image": img})
        except Exception as e:
            skipped.append((os.path.basename(p), str(e)))

    print(f"  Loaded {len(images)} images from {directory}")
    if skipped:
        print(f"  Skipped {len(skipped)} files:")
        for name, err in skipped:
            print(f"    {name}: {err}")
    return images

print("Loading Rutkowski portfolio...")
rutkowski_imgs = load_images(PORTFOLIO_DIR)

print("Loading Bowater (control) portfolio...")
bowater_imgs = load_images(CONTROL_DIR)

if not rutkowski_imgs:
    print("WARNING: No Rutkowski images found. Add images to assets/portfolio/")
if not bowater_imgs:
    print("WARNING: No Bowater images found. Add images to assets/control/")

Loading Rutkowski portfolio...
  Loaded 30 images from /content/drive/Othercomputers/laptop/research/diffusion_art/assets/portfolio
Loading Bowater (control) portfolio...
  Loaded 30 images from /content/drive/Othercomputers/laptop/research/diffusion_art/assets/control


---
## Section 2: Loss-Based Membership Inference

Computes denoising MSE at 6 timesteps for each portfolio image on both SD 1.5 and SD 2.1.
The key comparison: if Stability AI removed Rutkowski from SD 2.1's training data, his images
should have *higher* loss on SD 2.1 than SD 1.5. Bowater (in neither dataset) is the control.


In [21]:
  # Section 2: Loss-based MIA — SD 1.5 vs SD 2.1 on portfolio images
import gc
from diffusers import StableDiffusionPipeline, DDPMScheduler
from torchvision import transforms

MIA_TIMESTEPS = [50, 100, 200, 400, 600, 800]
img_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

def compute_mia_loss(pil_image, pipe, scheduler, timesteps, device):
    img_t = img_transform(pil_image).unsqueeze(0).to(device, dtype=torch.float16)
    with torch.no_grad():
        latent = pipe.vae.encode(img_t).latent_dist.mean * pipe.vae.config.scaling_factor
        null_ids = pipe.tokenizer("", return_tensors="pt", padding="max_length",
                                  max_length=pipe.tokenizer.model_max_length,
                                  truncation=True).input_ids.to(device)
        null_emb = pipe.text_encoder(null_ids).last_hidden_state
    # Normalise predictions to ε-space for both ε-prediction (SD 1.5) and
    # v-prediction (SD 2.1) models so losses are comparable across architectures.
    v_pred = (scheduler.config.prediction_type == "v_prediction")
    noise = torch.randn_like(latent)
    losses = []
    for t_val in timesteps:
        t = torch.tensor([t_val], device=device, dtype=torch.long)
        noisy = scheduler.add_noise(latent.float(), noise.float(), t).half()
        with torch.no_grad():
            pred = pipe.unet(noisy, t, encoder_hidden_states=null_emb).sample
        if v_pred:
            # Convert velocity → noise: ε = α_t · v + σ_t · x_t
            ac = scheduler.alphas_cumprod[t_val].to(device)
            alpha_t = ac ** 0.5
            sigma_t = (1 - ac) ** 0.5
            pred_eps = alpha_t * pred.float() + sigma_t * noisy.float()
        else:
            pred_eps = pred.float()
        losses.append(F.mse_loss(pred_eps, noise.float()).item())
    return losses

def run_mia(imgs, pipe, scheduler, label):
    out = []
    for i, item in enumerate(imgs):
        losses = compute_mia_loss(item["image"], pipe, scheduler, MIA_TIMESTEPS, device)
        out.append(losses)
        if (i + 1) % 5 == 0 or (i + 1) == len(imgs):
            print(f"  {label} [{i+1}/{len(imgs)}] mean={np.mean(losses):.4f}")
    return np.array(out)

# ── SD 1.5 ───────────────────────────────────────────────────────────────────
MODEL_ID_15 = "runwayml/stable-diffusion-v1-5"
print("Loading SD 1.5...")
pipe_15  = StableDiffusionPipeline.from_pretrained(MODEL_ID_15, torch_dtype=torch.float16,
               safety_checker=None, requires_safety_checker=False).to(device)
pipe_15.enable_attention_slicing()
sched_15 = DDPMScheduler.from_pretrained(MODEL_ID_15, subfolder="scheduler")

losses_15_r = run_mia(rutkowski_imgs, pipe_15, sched_15, "SD1.5 / Rutkowski")
losses_15_b = run_mia(bowater_imgs,   pipe_15, sched_15, "SD1.5 / Bowater")

del pipe_15; torch.cuda.empty_cache(); gc.collect()

# ── SD 2.1 ───────────────────────────────────────────────────────────────────
# Try full 768px model first; fall back to 512px base if repo is unavailable.
HF_TOKEN = os.environ.get('HF_TOKEN', '')
for MODEL_ID_21 in [
    "sd2-community/stable-diffusion-2-1",   # public community mirror
    "Manojb/stable-diffusion-2-1-base",     # public 512px base
    "stabilityai/stable-diffusion-2-1",     # original (requires license accept)
]:
    try:
        print(f"\nLoading {MODEL_ID_21}...")
        pipe_21 = StableDiffusionPipeline.from_pretrained(
            MODEL_ID_21, torch_dtype=torch.float16,
            safety_checker=None, requires_safety_checker=False,
            token=HF_TOKEN or None).to(device)
        break
    except Exception as e:
        print(f"  Failed ({e.__class__.__name__}): {e!s:.120}")
else:
    raise RuntimeError("Could not load any SD 2.x model — check HF token and network.")
pipe_21.enable_attention_slicing()
sched_21 = DDPMScheduler.from_pretrained(MODEL_ID_21, subfolder="scheduler")

losses_21_r = run_mia(rutkowski_imgs, pipe_21, sched_21, "SD2.1 / Rutkowski")
losses_21_b = run_mia(bowater_imgs,   pipe_21, sched_21, "SD2.1 / Bowater")

del pipe_21; torch.cuda.empty_cache(); gc.collect()

print("\nMean losses:")
print(f"  Rutkowski / SD 1.5: {losses_15_r.mean():.4f}")
print(f"  Rutkowski / SD 2.1: {losses_21_r.mean():.4f}  (gap: {losses_21_r.mean()-losses_15_r.mean():+.4f})")
print(f"  Bowater   / SD 1.5: {losses_15_b.mean():.4f}")
print(f"  Bowater   / SD 2.1: {losses_21_b.mean():.4f}  (gap: {losses_21_b.mean()-losses_15_b.mean():+.4f})")


Loading SD 1.5...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

  SD1.5 / Rutkowski [5/30] mean=0.2844
  SD1.5 / Rutkowski [10/30] mean=0.2371
  SD1.5 / Rutkowski [15/30] mean=0.3106
  SD1.5 / Rutkowski [20/30] mean=0.2885
  SD1.5 / Rutkowski [25/30] mean=0.2764
  SD1.5 / Rutkowski [30/30] mean=0.2689
  SD1.5 / Bowater [5/30] mean=0.2369
  SD1.5 / Bowater [10/30] mean=0.2878
  SD1.5 / Bowater [15/30] mean=0.1776
  SD1.5 / Bowater [20/30] mean=0.2144
  SD1.5 / Bowater [25/30] mean=0.2979
  SD1.5 / Bowater [30/30] mean=0.2821

Loading sd2-community/stable-diffusion-2-1...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

  SD2.1 / Rutkowski [5/30] mean=0.3099
  SD2.1 / Rutkowski [10/30] mean=0.2661
  SD2.1 / Rutkowski [15/30] mean=0.3346
  SD2.1 / Rutkowski [20/30] mean=0.3161
  SD2.1 / Rutkowski [25/30] mean=0.2995
  SD2.1 / Rutkowski [30/30] mean=0.2809
  SD2.1 / Bowater [5/30] mean=0.2680
  SD2.1 / Bowater [10/30] mean=0.3106
  SD2.1 / Bowater [15/30] mean=0.1878
  SD2.1 / Bowater [20/30] mean=0.2304
  SD2.1 / Bowater [25/30] mean=0.3223
  SD2.1 / Bowater [30/30] mean=0.3138

Mean losses:
  Rutkowski / SD 1.5: 0.2758
  Rutkowski / SD 2.1: 0.3003  (gap: +0.0245)
  Bowater   / SD 1.5: 0.2500
  Bowater   / SD 2.1: 0.2749  (gap: +0.0249)


In [ ]:
# Figure 2a: Loss vs timestep — side-by-side subplots, one per model
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["SD 1.5  (Rutkowski in training data)",
                    "SD 2.1  (Rutkowski reportedly excluded)"],
    shared_yaxes=False,
)

panels = [
    (1, losses_15_r, losses_15_b),
    (2, losses_21_r, losses_21_b),
]

for col, r_arr, b_arr in panels:
    for label, arr, color in [
        ("Greg Rutkowski", r_arr, "#dc2626"),
        ("Charlie Bowater (control)", b_arr, "#2563eb"),
    ]:
        m = arr.mean(axis=0)
        fig.add_trace(go.Scatter(
            x=MIA_TIMESTEPS, y=m, mode="lines+markers",
            name=label, legendgroup=label, showlegend=(col == 1),
            line=dict(color=color, width=2.5),
            marker=dict(size=7),
        ), row=1, col=col)

fig.update_xaxes(title_text="Diffusion timestep t")
fig.update_yaxes(title_text="Mean prediction MSE", col=1)
fig.update_layout(
    title="Figure 2a: Denoising Loss vs. Timestep",
    legend=dict(orientation="h", y=-0.18),
    font=dict(family="system-ui, sans-serif", size=12),
    height=420,
)
save_fig(fig, os.path.join(FIGURES_DIR))
fig.show()


In [ ]:
# Figures 2b and 2c: Mean loss per artist, one chart per model
gap_sd15 = losses_15_b.mean() - losses_15_r.mean()
gap_sd21 = losses_21_b.mean() - losses_21_r.mean()

results["sec3a"] = {
    "mean_loss_sd15_rutkowski": float(losses_15_r.mean()),
    "mean_loss_sd21_rutkowski": float(losses_21_r.mean()),
    "mean_loss_sd15_bowater":   float(losses_15_b.mean()),
    "mean_loss_sd21_bowater":   float(losses_21_b.mean()),
    "within_model_gap_sd15":    float(gap_sd15),
    "within_model_gap_sd21":    float(gap_sd21),
}

for fig_id, title, r_arr, b_arr, fname in [
    ("2b", f"Figure 2b: SD 1.5 — Mean Denoising Loss<br>"
           f"<sup>Gap (Bowater − Rutkowski): {gap_sd15:+.4f}</sup>",
     losses_15_r, losses_15_b, "fig2b_mia_sd15"),
    ("2c", f"Figure 2c: SD 2.1 — Mean Denoising Loss<br>"
           f"<sup>Gap (Bowater − Rutkowski): {gap_sd21:+.4f}</sup>",
     losses_21_r, losses_21_b, "fig2c_mia_sd21"),
]:
    fig = go.Figure()
    for label, arr, color in [
        ("Greg Rutkowski", r_arr, "#dc2626"),
        ("Charlie Bowater (control)", b_arr, "#2563eb"),
    ]:
        scores = arr.mean(axis=1)
        fig.add_trace(go.Bar(
            name=label,
            x=[label],
            y=[scores.mean()],
            error_y=dict(type="data", array=[scores.std()], visible=True),
            marker_color=color,
            width=0.4,
        ))
    fig.update_layout(
        title=title,
        yaxis_title="Mean MSE",
        showlegend=False,
        font=dict(family="system-ui, sans-serif", size=12),
        width=480, height=400,
        bargap=0.3,
    )
        save_fig(fig, os.path.join(FIGURES_DIR))
    fig.show()

print(f"SD 1.5 gap (Bowater − Rutkowski): {gap_sd15:+.4f}")
print(f"SD 2.1 gap (Bowater − Rutkowski): {gap_sd21:+.4f}")


---
## Section 6: Save All Results

In [24]:
# Save all numerical results to results.json
import json

# Convert numpy types for JSON serialization
def convert(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return obj

def deep_convert(obj):
    if isinstance(obj, dict):
        return {k: deep_convert(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [deep_convert(v) for v in obj]
    return convert(obj)

results_clean = deep_convert(results)
results_path = os.path.join(RESULTS_DIR, "results.json")

with open(results_path, "w") as f:
    json.dump(results_clean, f, indent=2)

print(f"Results saved to: {results_path}")
print(json.dumps(results_clean, indent=2))

Results saved to: /content/drive/Othercomputers/laptop/research/diffusion_art/results/results.json
{
  "sec3a": {
    "mean_loss_sd15_rutkowski": 0.27580782338562937,
    "mean_loss_sd21_rutkowski": 0.30026394756924774,
    "mean_loss_sd15_bowater": 0.249999174884417,
    "mean_loss_sd21_bowater": 0.27487145787518885,
    "within_model_gap_sd15": -0.025808648501212367,
    "within_model_gap_sd21": -0.025392489694058884
  }
}


In [25]:
print("=" * 60)
print("MIA RESULTS")
print("=" * 60)
r = results.get("sec3a", {})
print(f"  Rutkowski / SD 1.5: {r.get('mean_loss_sd15_rutkowski', 'N/A'):.4f}")
print(f"  Rutkowski / SD 2.1: {r.get('mean_loss_sd21_rutkowski', 'N/A'):.4f}")
print(f"  Bowater   / SD 1.5: {r.get('mean_loss_sd15_bowater',   'N/A'):.4f}")
print(f"  Bowater   / SD 2.1: {r.get('mean_loss_sd21_bowater',   'N/A'):.4f}")
print(f"  Within-model gap SD 1.5: {r.get('within_model_gap_sd15', 'N/A'):+.4f}")
print(f"  Within-model gap SD 2.1: {r.get('within_model_gap_sd21', 'N/A'):+.4f}")


MIA RESULTS
  Rutkowski / SD 1.5: 0.2758
  Rutkowski / SD 2.1: 0.3003
  Bowater   / SD 1.5: 0.2500
  Bowater   / SD 2.1: 0.2749
  Within-model gap SD 1.5: -0.0258
  Within-model gap SD 2.1: -0.0254
